# NB08 — Slide Embeddings Export

Loads the final MILTransformer weights from NB06 and exports a 768-d slide embedding per slide using max pooling over the transformer output. Routes embeddings into per-dataset subdirectories under `embeddings/` based on the slide manifests for downstream evaluation. Falls back to raw feature mean if no trained checkpoint is available.

In [ ]:
import os, sys, json, time, gc
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

WORKSPACE = Path(os.environ.get('WORKSPACE', './workspace'))
EMB_DIR     = WORKSPACE / 'embeddings'
MANIFESTS   = WORKSPACE / 'manifests'
DIAG        = WORKSPACE / 'diagnostics'
FEAT05      = WORKSPACE / 'features' / 'scale0p5'
FEAT20      = WORKSPACE / 'features' / 'scale2p0'
WEIGHTS_DIR = WORKSPACE / 'weights'
for p in [EMB_DIR, DIAG]:
    p.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from safetensors.torch import load_file as load_safetensors

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def load_slide_sets():
    sets = {}
    def _load_csv(name):
        p = MANIFESTS / f'manifest_{name}.csv'
        if p.exists():
            df = pd.read_csv(p)
            if 'slide_id' in df.columns:
                return set(df['slide_id'])
            elif 'filename' in df.columns:
                return set(Path(x).stem for x in df['filename'])
        return set()
    sets['tcga']        = _load_csv('tcga')
    sets['camelyon16']  = _load_csv('camelyon16')
    sets['camelyon17']  = _load_csv('camelyon17')
    return sets

SLIDESETS = load_slide_sets()

def available_two_scale_ids():
    s05 = set(p.stem for p in FEAT05.glob('*.npy'))
    s20 = set(p.stem for p in FEAT20.glob('*.npy'))
    return sorted(list(s05 & s20))

TWO_SCALE_IDS = available_two_scale_ids()

def dataset_of(slide_id):
    if slide_id in SLIDESETS.get('camelyon16', set()): return 'CAMELYON16'
    if slide_id in SLIDESETS.get('camelyon17', set()): return 'CAMELYON17'
    if slide_id in SLIDESETS.get('tcga', set()):       return 'TCGA'
    return 'OTHER'

class PositionalEncoder(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(3, d_model//2), nn.GELU(), nn.Linear(d_model//2, d_model))
    def forward(self, mmxy, scale_um):
        x = torch.cat([mmxy, scale_um], dim=-1)
        return self.proj(x)

class MILTransformer(nn.Module):
    def __init__(self, d_model=768, n_heads=8, n_layers=6, ff_mult=4, dropout=0.1):
        super().__init__()
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=int(ff_mult * d_model),
            dropout=dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.ln  = nn.LayerNorm(d_model)
        self.pos = PositionalEncoder(d_model)
        self.proj_global = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, d_model))
        self.pred_global = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, d_model))
    def forward(self, feats, mmxy, scale_um, pad_mask):
        pos = self.pos(mmxy, scale_um)
        x = feats + pos
        x = self.enc(x, src_key_padding_mask=pad_mask)
        x = self.ln(x)
        x_for_pool = x.masked_fill(pad_mask.unsqueeze(-1), float('-inf'))
        g = x_for_pool.max(dim=1).values
        g_proj = self.proj_global(g)
        return g_proj, self.pred_global(g_proj), x

def load_meta(p: Path) -> pd.DataFrame:
    df = pd.read_parquet(p)
    cols_lower = {c.lower(): c for c in df.columns}
    def pick(*names):
        for n in names:
            if n in df.columns: return n
            if n.lower() in cols_lower: return cols_lower[n.lower()]
        raise KeyError(f'missing one of {names} in {p.name}')
    xcol = pick('x'); ycol = pick('y'); lvlcol = pick('level', 'lvl'); sccol = pick('scale_um_per_px')
    tsize = 256
    for n in ('tile_size', 'tile_px', 'size'):
        if n in df.columns:
            try: tsize = int(df[n].iloc[0])
            except Exception: pass
            break
    out = df[[xcol, ycol, lvlcol, sccol]].copy()
    out.columns = ['x', 'y', 'level', 'scale_um_per_px']
    out['tile_px'] = tsize
    return out

def compute_mm_xy(df: pd.DataFrame) -> np.ndarray:
    um_per_px = df['scale_um_per_px'].astype(float).to_numpy()
    mm_per_px = um_per_px / 1000.0
    cx = (df['x'].to_numpy() + df['tile_px'].to_numpy()/2.0) * mm_per_px
    cy = (df['y'].to_numpy() + df['tile_px'].to_numpy()/2.0) * mm_per_px
    return np.stack([cx, cy], axis=1).astype(np.float32)

def load_trained_model():
    txt = WEIGHTS_DIR / 'latest.txt'
    if not txt.exists():
        print('[WARN] no trained checkpoint found; using raw feature mean fallback'); return None
    ckpt_name = txt.read_text(encoding='utf-8').strip()
    ckpt_path = WEIGHTS_DIR / ckpt_name
    if not ckpt_path.exists():
        print(f'[WARN] checkpoint {ckpt_path} not found; falling back to raw mean'); return None
    model = MILTransformer(d_model=768, n_heads=8, n_layers=6, ff_mult=4, dropout=0.1).to(DEVICE)
    sd = load_safetensors(str(ckpt_path))
    model.load_state_dict(sd, strict=True)
    model.eval()
    print(f'[OK] loaded MILTransformer from {ckpt_path.name}')
    return model

def embed_one_with_model(slide_id, model):
    f05_path = FEAT05 / f'{slide_id}.npy'
    f20_path = FEAT20 / f'{slide_id}.npy'
    if not (f05_path.exists() and f20_path.exists()):
        return {'slide_id': slide_id, 'ok': False, 'reason': 'missing_feature_file'}
    try:
        a = np.load(f05_path, mmap_mode='r').astype(np.float32)
        b = np.load(f20_path, mmap_mode='r').astype(np.float32)
        if a.ndim != 2 or b.ndim != 2 or a.shape[1] != 768 or b.shape[1] != 768:
            return {'slide_id': slide_id, 'ok': False, 'reason': f'bad_shape a{tuple(a.shape)} b{tuple(b.shape)}'}
        rng = np.random.default_rng(1337 + (hash(slide_id) % (2**16)))
        if a.shape[0] > 1200:
            idx = rng.choice(a.shape[0], 1200, replace=False); a = a[idx]
        if b.shape[0] > 400:
            idx = rng.choice(b.shape[0], 400, replace=False); b = b[idx]
        feats = np.concatenate([a, b], axis=0)

        if model is not None:
            T = feats.shape[0]
            meta05_path = FEAT05 / f'{slide_id}_meta.parquet'
            meta20_path = FEAT20 / f'{slide_id}_meta.parquet'
            if meta05_path.exists() and meta20_path.exists():
                m05 = load_meta(meta05_path).iloc[:a.shape[0]]
                m20 = load_meta(meta20_path).iloc[:b.shape[0]]
                mmxy = np.concatenate([compute_mm_xy(m05), compute_mm_xy(m20)], axis=0)
            else:
                mmxy = np.zeros((T, 2), dtype=np.float32)
            scl = np.concatenate([
                np.full((a.shape[0], 1), 0.5, dtype=np.float32),
                np.full((b.shape[0], 1), 2.0, dtype=np.float32),
            ], axis=0)
            feats_t = torch.from_numpy(feats).unsqueeze(0).to(DEVICE)
            mmxy_t  = torch.from_numpy(mmxy).unsqueeze(0).to(DEVICE)
            scl_t   = torch.from_numpy(scl).unsqueeze(0).to(DEVICE)
            pad_t   = torch.zeros(1, T, dtype=torch.bool, device=DEVICE)
            with torch.no_grad():
                g_proj, _, _ = model(feats_t, mmxy_t, scl_t, pad_t)
            emb = g_proj.squeeze(0).cpu().numpy().astype(np.float32)
        else:
            emb = feats.mean(axis=0).astype(np.float32)

        ds = dataset_of(slide_id)
        out_dir = EMB_DIR / ds
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f'{slide_id}.npy'
        np.save(out_path, emb)
        return {
            'slide_id': slide_id, 'dataset': ds, 'ok': True,
            'path_emb': str(out_path),
            't05': int(a.shape[0]), 't20': int(b.shape[0]),
            'norm': float(np.linalg.norm(emb)),
        }
    except Exception as e:
        return {'slide_id': slide_id, 'ok': False, 'reason': f'{type(e).__name__}:{e}'}

print('NB08: slide embeddings export using trained transformer')
model = load_trained_model()
target_ids = TWO_SCALE_IDS
print(f'[PLAN] slides with 2-scale features: {len(target_ids)}')
if not target_ids:
    print('[EXIT] nothing to export'); raise SystemExit(0)

t0 = time.time(); rows = []
ok = 0; bad = 0
for i, sid in enumerate(target_ids, 1):
    r = embed_one_with_model(sid, model)
    rows.append(r)
    ok += int(r.get('ok', False))
    bad += int(not r.get('ok', False))
    if i % 200 == 0:
        print(f'[{i:6d}/{len(target_ids)}] ok={ok} bad={bad}')

df = pd.DataFrame(rows)
for ds, g in df[df['ok'] == True].groupby('dataset'):
    out_csv = EMB_DIR / f'{ds.lower()}_index.csv'
    g[['slide_id', 'path_emb', 't05', 't20', 'norm']].to_csv(out_csv, index=False)
    print(f'[OK] index for {ds}: {len(g)} -> {out_csv}')

print(f'\n[DONE] NB08 complete. ok={ok} bad={bad}. Next: NB09 (CAMELYON17 LOCO).')